 best-parameter repeat check

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed
from tqdm.auto import tqdm

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'src').exists():
            return candidate
    return start

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'ROOT = {ROOT}')

ROOT = /home/zhuran/ran/CategoryLearning


/home/zhuran/.conda/envs/cate_learn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.Bayesian_state.utils.optimizer_common import evaluate_state_model_run
from src.Bayesian_state.utils.optimizer_grid import StateModelGridOptimizer

CONFIG_PATH = ROOT / 'configs/grid_opt_cfg/pmh_cond1.yaml'
RESULTS_DIR = ROOT / 'results/state-based-grid-result/pmh/cond1'
DATA_PATH = ROOT / 'data/processed/Task2_processed.csv'
PLOTS_DIR = ROOT / 'notebooks' / 'cond1_repeat_best_params_accuracy'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    grid_cfg = yaml.safe_load(f) or {}

engine_config_path = grid_cfg.get('engine_config_path')
if engine_config_path is None:
    raise ValueError('pmh_cond1.yaml must define engine_config_path')
engine_config_file = (CONFIG_PATH.parent / engine_config_path).resolve()
with engine_config_file.open('r', encoding='utf-8') as f:
    engine_config = yaml.safe_load(f) or {}

with (RESULTS_DIR / 'all_subjects.json').open('r', encoding='utf-8') as f:
    subject_results = json.load(f)

subject_ids = sorted(int(k) for k in subject_results.keys())
window_size = grid_cfg.get('window_size', 16)
if isinstance(window_size, list):
    window_size = int(window_size[0])
else:
    window_size = int(window_size)

stop_at = float(grid_cfg.get('stop_at', 1.0))
max_trials = grid_cfg.get('max_trials')
max_trials = None if max_trials is None else int(max_trials)

N_REPEATS = 32
REPEAT_N_JOBS = max(1, min(8, os.cpu_count() or 1))

optimizer = StateModelGridOptimizer(
    engine_config=engine_config,
    processed_data_dir=DATA_PATH.parent,
    n_jobs=1,
)
optimizer.prepare_data(DATA_PATH)

print(f'Subjects: {subject_ids}')
print(f'window_size = {window_size}, stop_at = {stop_at}, max_trials = {max_trials}')
print(f'N_REPEATS = {N_REPEATS}, REPEAT_N_JOBS = {REPEAT_N_JOBS}')
print(f'Plots will be saved to: {PLOTS_DIR}')

INFO:cat-learning:logger is running normally.


/home/zhuran/ran/CategoryLearning/logs/Run_20260402_095734.log
Subjects: [125, 126, 127, 128, 129, 130, 131, 132]
window_size = 16, stop_at = 1.0, max_trials = None
N_REPEATS = 32, REPEAT_N_JOBS = 8
Plots will be saved to: /home/zhuran/ran/CategoryLearning/notebooks/cond1_repeat_best_params_accuracy


In [3]:
def get_subject_inputs(subject_id: int):
    subject_frame = optimizer._get_subject_frame(subject_id, stop_at)
    arrays = optimizer._extract_arrays(subject_frame, max_trials)
    condition = int(subject_frame['condition'].iloc[0])
    best_params = dict(subject_results[str(subject_id)]['best_params'])
    return condition, arrays, best_params

def run_repeats_for_subject(subject_id: int, n_repeats: int = N_REPEATS):
    condition, arrays, best_params = get_subject_inputs(subject_id)

    runs = Parallel(n_jobs=REPEAT_N_JOBS)(
        delayed(evaluate_state_model_run)(
            subject_id=subject_id,
            condition=condition,
            arrays=arrays,
            params=best_params,
            engine_config_template=engine_config,
            processed_data_dir=DATA_PATH.parent,
            window_size=window_size,
            keep_logs=True,
            include_step_log=False,
        )
        for _ in tqdm(range(n_repeats), desc=f'Subject {subject_id}')
    )
    return runs, best_params

def plot_subject_accuracy(subject_id: int, runs, best_params: dict, save_path: Path | None = None):
    true_acc = np.asarray(runs[0].metrics['sliding_true_acc'], dtype=float)
    pred_stack = np.vstack([np.asarray(run.metrics['sliding_pred_acc'], dtype=float) for run in runs])
    pred_mean = np.nanmean(pred_stack, axis=0)
    pred_std = np.nanstd(pred_stack, axis=0)
    mean_error = np.asarray([run.mean_error for run in runs], dtype=float)

    x = np.arange(1, len(true_acc) + 1)
    fig, ax = plt.subplots(figsize=(10, 4))

    for pred in pred_stack:
        ax.plot(x, pred, color='#d95f02', alpha=0.12, lw=1)

    ax.plot(x, true_acc, color='#1f77b4', lw=2.2, label='True')
    ax.plot(x, pred_mean, color='#d95f02', lw=2.2, label='Predicted mean')
    ax.fill_between(x, pred_mean - pred_std, pred_mean + pred_std, color='#d95f02', alpha=0.18, label='Predicted ±1 std')

    gamma = best_params.get('gamma')
    w0 = best_params.get('w0')
    ax.set_title(f'Subject {subject_id} | gamma={gamma}, w0={w0} | repeats={len(runs)}')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Sliding accuracy')
    ax.set_xlim(1, len(true_acc))
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, loc='lower right')

    text = f'mean_error = {float(np.mean(mean_error)):.4f} ± {float(np.std(mean_error)):.4f}'
    ax.text(0.01, 0.04, text, transform=ax.transAxes, fontsize=10, va='bottom')

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return fig, {
        'subject_id': subject_id,
        'best_params': best_params,
        'mean_error_mean': float(np.mean(mean_error)),
        'mean_error_std': float(np.std(mean_error)),
        'pred_curve_mean': pred_mean,
        'pred_curve_std': pred_std,
        'true_curve': true_acc,
    }

def plot_subject_on_axis(ax, subject_id: int, runs, best_params: dict):
    true_acc = np.asarray(runs[0].metrics['sliding_true_acc'], dtype=float)
    pred_stack = np.vstack([np.asarray(run.metrics['sliding_pred_acc'], dtype=float) for run in runs])
    pred_mean = np.nanmean(pred_stack, axis=0)
    pred_std = np.nanstd(pred_stack, axis=0)
    x = np.arange(1, len(true_acc) + 1)

    ax.plot(x, true_acc, color='#1f77b4', lw=1.8, label='True')
    ax.plot(x, pred_mean, color='#d95f02', lw=1.8, label='Predicted mean')
    ax.fill_between(x, pred_mean - pred_std, pred_mean + pred_std, color='#d95f02', alpha=0.18)
    ax.set_title(f'Subject {subject_id}')
    ax.set_xlim(1, len(true_acc))
    ax.set_ylim(0.0, 1.05)
    ax.grid(alpha=0.2)
    ax.text(0.02, 0.05, f"g={best_params.get('gamma')}, w0={best_params.get('w0')}", transform=ax.transAxes, fontsize=9)

    return ax

In [5]:
repeat_results = {}
summary_rows = []

for subject_id in tqdm(subject_ids, desc='Subjects'):
    runs, best_params = run_repeats_for_subject(subject_id, N_REPEATS)
    repeat_results[subject_id] = {'runs': runs, 'best_params': best_params}

    save_path = PLOTS_DIR / f'subject_{subject_id}_repeat_accuracy.png'


    mean_errors = np.asarray([run.mean_error for run in runs], dtype=float)
    pred_stack = np.vstack([np.asarray(run.metrics['sliding_pred_acc'], dtype=float) for run in runs])
    summary_rows.append({
        'subject_id': subject_id,
        'gamma': best_params.get('gamma'),
        'w0': best_params.get('w0'),
        'repeat_mean_error': float(np.mean(mean_errors)),
        'repeat_std_error': float(np.std(mean_errors)),
        'pred_curve_mean_std': float(np.nanmean(np.nanstd(pred_stack, axis=0))),
        'n_unique_mean_errors': int(len(np.unique(np.round(mean_errors, 8)))),
        'plot_path': str(save_path),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('subject_id').reset_index(drop=True)
summary_df

Subjects:   0%|          | 0/8 [00:00<?, ?it/s]INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:216: RuntimeWarning: divide by zero encountered in log
  return np.log(clipped)
INFO:cat-learning:  - Module 'memory_mod' registered as 'self.memory_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/p

{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}}, 'hypo_transitions_mod': {'class': 'src.Bayesian_state.problems.modules.hypo_transitions.DynamicHypothesisModule', 'kwargs': {'strategies': 'original_strategies_a', 'init_num': 3}}, 'likelihood_mod': {'class': 'src.Bayesian_state.problems.modules.likelihood.LikelihoodModule', 'kwargs': {}}, 'memory_mod': {'class': 'src.Bayesian_state.problems.modules.memory.DualMemoryModule', 'kwargs': {'w0': 1.0, 'gamma': 0.995}}}
name: perception_mod mod_kwargs: {}
{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta

INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:216: RuntimeWarning: divide by zero encountered in log
  return np.log(clipped)
INFO:cat-learning:  - Module 'memory_mod' registered as 'self.memory_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:337: RuntimeWarning: invalid value encountered in multiply
  log_posterior = self.w0 * self.state["static"] + (1 - self.w0) * self.state["fade"]
INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' re

name: beta_mod mod_kwargs: {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}
name: hypo_transitions_mod mod_kwargs: {'strategies': 'original_strategies_a', 'init_num': 3}
name: likelihood_mod mod_kwargs: {}
name: memory_mod mod_kwargs: {'w0': 1.0, 'gamma': 0.995}
name: beta_mod mod_kwargs: {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}
name: hypo_transitions_mod mod_kwargs: {'strategies': 'original_strategies_a', 'init_num': 3}
name: likelihood_mod mod_kwargs: {}
name: memory_mod mod_kwargs: {'w0': 1.0, 'gamma': 0.995}
name: beta_mod mod_kwargs: {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}
name: hypo_transitions_mod mod_kwargs: {'strategies': 'original_strategies_a', 'init_num'

INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:216: RuntimeWarning: divide by zero encountered in log
  return np.log(clipped)
INFO:cat-learning:  - Module 'memory_mod' registered as 'self.memory_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:337: RuntimeWarning: invalid value encountered in multiply
  log_posterior = self.w0 * self.state["static"] + (1 - self.w0) * self.state["fade"]
INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' re

...

{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}}, 'hypo_transitions_mod': {'class': 'src.Bayesian_state.problems.modules.hypo_transitions.DynamicHypothesisModule', 'kwargs': {'strategies': 'original_strategies_a', 'init_num': 3}}, 'likelihood_mod': {'class': 'src.Bayesian_state.problems.modules.likelihood.LikelihoodModule', 'kwargs': {}}, 'memory_mod': {'class': 'src.Bayesian_state.problems.modules.memory.DualMemoryModule', 'kwargs': {'w0': 0.3, 'gamma': 0.999}}}
name: perception_mod mod_kwargs: {}
{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta

Subject 132: 100%|██████████| 32/32 [00:01<00:00, 24.73it/s]

{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta_min': 0.1, 'beta_max': 50.0, 'decrease_rate': 0.2, 'correct_additive': 1, 'use_prior_scaling': True, 'prior_beta_scale': 10.0}}, 'hypo_transitions_mod': {'class': 'src.Bayesian_state.problems.modules.hypo_transitions.DynamicHypothesisModule', 'kwargs': {'strategies': 'original_strategies_a', 'init_num': 3}}, 'likelihood_mod': {'class': 'src.Bayesian_state.problems.modules.likelihood.LikelihoodModule', 'kwargs': {}}, 'memory_mod': {'class': 'src.Bayesian_state.problems.modules.memory.DualMemoryModule', 'kwargs': {'w0': 0.3, 'gamma': 0.999}}}
name: perception_mod mod_kwargs: {}
{'perception_mod': {'class': 'src.Bayesian_state.problems.modules.perception.PerceptionModule'}, 'beta_mod': {'class': 'src.Bayesian_state.problems.modules.beta.BetaModule', 'kwargs': {'beta_init': 1.0, 'beta


INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:216: RuntimeWarning: divide by zero encountered in log
  return np.log(clipped)
INFO:cat-learning:  - Module 'memory_mod' registered as 'self.memory_mod'.
INFO:cat-learning:  - Module 'perception_mod' registered as 'self.perception_mod'.
INFO:cat-learning:  - Module 'beta_mod' registered as 'self.beta_mod'.
INFO:cat-learning:  - Module 'hypo_transitions_mod' registered as 'self.hypo_transitions_mod'.
INFO:cat-learning:  - Module 'likelihood_mod' registered as 'self.likelihood_mod'.
/home/zhuran/ran/CategoryLearning/src/Bayesian_state/problems/modules/memory.py:216: RuntimeWarning:

,subject_id,gamma,w0,repeat_mean_error,repeat_std_error,pred_curve_mean_std,n_unique_mean_errors,plot_path
0,125,0.995,1.000,0.198019,0.085603,0.143899,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
1,126,0.995,0.150,0.223223,0.077342,0.134840,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
2,127,0.990,0.150,0.143390,0.040048,0.145752,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
3,128,0.100,1.000,0.171314,0.035258,0.153740,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
4,129,0.900,0.500,0.205092,0.028582,0.180983,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
5,130,0.995,1.000,0.158470,0.021846,0.135823,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
6,131,0.950,0.005,0.177892,0.051810,0.169047,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
7,132,0.999,0.300,0.178178,0.036291,0.130562,32,/home/zhuran/ran/CategoryLearning/notebooks/co...


In [6]:
def plot_subject_repeat_panels(subject_id: int, runs, best_params: dict, save_path: Path | None = None):
    n_runs = len(runs)
    ncols = 4
    nrows = int(np.ceil(n_runs / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.2 * nrows), sharex=False, sharey=True)
    axes = np.atleast_1d(axes).ravel()

    gamma = best_params.get('gamma')
    w0 = best_params.get('w0')

    for idx, (ax, run) in enumerate(zip(axes, runs), start=1):
        true_acc = np.asarray(run.metrics['sliding_true_acc'], dtype=float)
        pred_acc = np.asarray(run.metrics['sliding_pred_acc'], dtype=float)
        x = np.arange(1, len(true_acc) + 1)

        ax.plot(x, true_acc, color='#1f77b4', lw=1.8, label='True')
        ax.plot(x, pred_acc, color='#d95f02', lw=1.8, label='Predicted')
        ax.set_title(f'Repeat {idx}')
        ax.set_xlim(1, len(true_acc))
        ax.set_ylim(0.0, 1.05)
        ax.grid(alpha=0.2)
        ax.text(0.02, 0.06, f'mean_error={float(run.mean_error):.4f}', transform=ax.transAxes, fontsize=9)

    for ax in axes[n_runs:]:
        ax.axis('off')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
    fig.suptitle(f'Cond 1 subject {subject_id} best-parameter repeat accuracy | g={gamma}, w0={w0}', y=1.01, fontsize=14)
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
    return fig

subject_plot_paths = {}
for subject_id in tqdm(subject_ids, desc='Render subjects'):
    runs = repeat_results[subject_id]['runs']
    best_params = repeat_results[subject_id]['best_params']
    save_path = PLOTS_DIR / f'subject_{subject_id}_{N_REPEATS}repeat_panels.png'
    fig = plot_subject_repeat_panels(subject_id, runs, best_params, save_path=save_path)
    plt.close(fig)
    subject_plot_paths[subject_id] = str(save_path)

summary_df['plot_path'] = summary_df['subject_id'].map(subject_plot_paths)
summary_df

Render subjects: 100%|██████████| 8/8 [00:18<00:00,  2.26s/it]


,subject_id,gamma,w0,repeat_mean_error,repeat_std_error,pred_curve_mean_std,n_unique_mean_errors,plot_path
0,125,0.995,1.000,0.198019,0.085603,0.143899,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
1,126,0.995,0.150,0.223223,0.077342,0.134840,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
2,127,0.990,0.150,0.143390,0.040048,0.145752,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
3,128,0.100,1.000,0.171314,0.035258,0.153740,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
4,129,0.900,0.500,0.205092,0.028582,0.180983,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
5,130,0.995,1.000,0.158470,0.021846,0.135823,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
6,131,0.950,0.005,0.177892,0.051810,0.169047,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
7,132,0.999,0.300,0.178178,0.036291,0.130562,32,/home/zhuran/ran/CategoryLearning/notebooks/co...
